In [17]:
from promenade.models import *
from promenade.agents import build_graph, build_reranker_graph
from langchain.tools import tool
from langchain.agents import create_agent

In [18]:
model = llm
reranker = RetreiveReranker(
    rerank_n=1000, 
    retrieve_n=10000, 
    rerank_model=RERANKER_MODEL)
WEB_TOOLS = WebTools(READER_URL)
DEV = False
SAMPLES_PATH = PROJECT_ROOT / "docs" / "samples"
base_url = "https://kosmo-museum.ru/" 
graph = build_graph(WEB_TOOLS, reranker, model, DEV=DEV, SAMPLES_PATH=SAMPLES_PATH)
reranker_graph = build_reranker_graph(
    llm=llm
)

In [19]:
import json

@tool
def parse_and_insert_into_db(url: HttpUrl) -> str:
    """Parse web page and insert into DB.

    Args:
        url (HttpUrl): URL address of the page to parse.

    Returns:
        str: JSON string with keys:
            - status: "success" or "error"
            - message: human-readable description
            - details: additional context
    """
    try:
        state = asyncio.run(graph.ainvoke({"url": str(url), "subdocs": [], "results": []}))
        count  = sum(1 for r in state["results"] if r["ok"])
        result: ToolResult = {
            "status": "success", 
            "message": f"Sucssefully created {count} entities in the sql database and vesctor database.", 
            "details": {
                "count": count, 
                "result": state["results"]
            }
        }
        return json.dumps(result, ensure_ascii=False)

    except Exception as e:
        result: ToolResult = {
            "status": "error", 
            "message": str(e), 
            "details": {}
        }
        return json.dumps(result, ensure_ascii=False)
    

@tool
def extract_from_vector_stor(query: str) -> str:
    """Retrieve information about venues from Qdrant database.

    Args:
        query (str): User query in Russian. Passed exactly as-is to the database for vector search.

    Returns:
        str: JSON string with keys:
            - status: "success" or "error"
            - message: human-readable description
            - details: additional context with search results
    """
    try:
        state = asyncio.run(reranker_graph.ainvoke(
            {
                "input_query": query,
                "reranked_documents": None,
                "retrieved_count": None,
                "result": None
            }
        ))
        result: ToolResult = {
            "status": "success", 
            "message": f"Sucssefully retrieved information about {state["retrieved_count"]} places", 
            "details": {
                "result": state["result"]
            }
        }
        return json.dumps(result, ensure_ascii=False)
    except Exception as e:
        result: ToolResult = {
            "status": "error", 
            "message": str(e), 
            "details": {}
        }
        return json.dumps(result, ensure_ascii=False)

In [20]:
GENERAL_AGENT_SYSTEM_PROMPT = """
You are Promenade — an AI assistant for discovering cultural leisure activities in Moscow.

## Core Functionality
Promenade crawls venue pages (museums, exhibitions, concerts, festivals), extracts structured information,
persists data to SQL database and Qdrant vector store, and answers natural-language queries through retrieval-augmented generation.

## Available Modes (Agent Modes)
The agent supports multiple operating modes, selected based on user input or context:

### 1. Crawl & Store Mode (current)
When the user provides a URL to a venue page:
- Use `parse_and_insert_into_db` tool to crawl and store venue information
- Extracts schedule, ticket prices, exhibitions, and contact details
- Stores structured data in SQL database (`museum`, `schedule` tables)
- Creates vector embeddings in Qdrant for semantic search
- After getting a result of the tool only answer the user about quantity of entities created in database and their names.
- Do not suggest further interaction with user

### 2. Query & Retrieve Mode (current)
When the user asks natural-language questions about venues:
- Use `extract_from_vector_stor` tool to extract information from Qdrant database
- Use retrieval from Qdrant to find relevant venues
- Generate answers using LLM with retrieved context

### 3. Search Mode (future)
When the user wants to browse venues by category:
- Filter venues by type (museum, concert, festival, etc.)
- Filter by schedule (open now, weekend, specific date)
- Filter by price range or special offers

## Guidelines
- In Crawl & Store mode, only use the provided URL (do not search for others)
- In Query mode, always cite sources from the retrieved data
- When uncertain, ask clarifying questions rather than making assumptions

## Output Format
- Use Russian for all responses unless user specifies otherwise
"""

agent = create_agent(
    model=model,
    tools=[parse_and_insert_into_db, extract_from_vector_stor],
    system_prompt=GENERAL_AGENT_SYSTEM_PROMPT,
)

In [21]:
# agent_result = agent.invoke(
#     {"messages": [{"role": "user", "content": "Хочу как-нибудь сходить в https://pushkinmuseum.art/"}]},
# )
# print(agent_result)

In [22]:
agent_result = agent.invoke(
    {"messages": [{"role": "user", "content": "Как работает пушкинский музей"}]},
)
print(agent_result)

100%|██████████| 2/2 [00:00<00:00,  2.99it/s]


{'messages': [HumanMessage(content='Как работает пушкинский музей', additional_kwargs={}, response_metadata={}, id='f877f5f3-a632-40d2-8904-ad4325123daf'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 30, 'prompt_tokens': 758, 'total_tokens': 788, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 528}}, 'model_provider': 'openai', 'model_name': 'Qwen/Qwen3-235B-A22B-Instruct-2507', 'system_fingerprint': None, 'id': 'chatcmpl-16841bde-b7dc-42e2-a9ce-ebc69a4c259c', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019df94a-be48-71a0-ba96-7d24d99838ee-0', tool_calls=[{'name': 'extract_from_vector_stor', 'args': {'query': 'Как работает пушкинский музей'}, 'id': 'chatcmpl-tool-ca66878e073f472d81298adf0ed54bd2', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 758, 'output_tokens': 30, 'total_tokens': 788, 'input_token_details': {'ca

In [23]:
agent_result

{'messages': [HumanMessage(content='Как работает пушкинский музей', additional_kwargs={}, response_metadata={}, id='f877f5f3-a632-40d2-8904-ad4325123daf'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 30, 'prompt_tokens': 758, 'total_tokens': 788, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 528}}, 'model_provider': 'openai', 'model_name': 'Qwen/Qwen3-235B-A22B-Instruct-2507', 'system_fingerprint': None, 'id': 'chatcmpl-16841bde-b7dc-42e2-a9ce-ebc69a4c259c', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019df94a-be48-71a0-ba96-7d24d99838ee-0', tool_calls=[{'name': 'extract_from_vector_stor', 'args': {'query': 'Как работает пушкинский музей'}, 'id': 'chatcmpl-tool-ca66878e073f472d81298adf0ed54bd2', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 758, 'output_tokens': 30, 'total_tokens': 788, 'input_token_details': {'

In [24]:
print(agent_result['messages'][-2].content)
print(agent_result['messages'][-1].content)

{"status": "success", "message": "Sucssefully retrieved information about 2 places", "details": {"result": "Главное здание. Base url: https://pushkinmuseum.art/ \n\n## Главное здание\n**Часы работы**  \n- Вторник: 10:00–21:00  \n- Среда: 10:00–19:00  \n- Четверг: 10:00–21:00  \n- Пятница: 11:00–21:00  \n- Суббота: 10:00–21:00  \n- Воскресенье: 10:00–19:00  \n- Понедельник: выходной  \n\n**Стоимость билетов**  \n— информация отсутствует —  \n\n**Актуальные выставки и события**  \n- **Князья Барятинские. Искусство для наследников** (21.04.2026 – 05.07.2026): выставка, посвящённая истории и коллекциям княжеского рода.  \n- **Ларионов / Гончарова. Начало** (31.03.2026 – 28.06.2026): экспозиция ранних работ художников в зале 31.  \n- **«Московский Витрувий» в ГМИИ им. А.С. Пушкина** (27.01.2026 – 31.05.2026): выставка в зале 7, посвящённая архитектуре.  \n\n**Специальные предложения**  \n- Режим работы изменён в майские праздники (подробности на сайте).  \n- Возможны особенности работы касс